In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

In [2]:
x1 = np.array([1, 2, 3])
x2 = 2*x1

y = np.array([4, 6, 8])

In [3]:
y

array([4, 6, 8])

In [5]:
all_ones = np.ones(x1.shape[0])
X = np.array([all_ones, x1, x2]).T
X.shape

(3, 3)

In [6]:
X

array([[1., 1., 2.],
       [1., 2., 4.],
       [1., 3., 6.]])

In [7]:
def solve_normal_equation(X, y):
    try:
        theta = np.linalg.inv(X.T @ X) @ X.T @ y
        return theta
    except np.linalg.LinAlgError:
        print('The matrix is singular')
        print("X.T @ X = \n", X.T @ X)
        return None

In [8]:
solve_normal_equation(X, y)

The matrix is singular
X.T @ X = 
 [[ 3.  6. 12.]
 [ 6. 14. 28.]
 [12. 28. 56.]]


In [9]:
np.linalg.matrix_rank(X), np.linalg.matrix_rank(X.T @ X)

(2, 2)

As we can see above, X which is the feature matrix is singular and has rank 2 (full rank = 3), and hence X is not invertible and we cannot calculate the  value of the optimal parameters or solve the normal equation of X and y. This is a problem that arises when we have a multicollinearity in our feature matrix. Multicollinearity is a phenomenon in which one feature can be linearly predicted from the others with a substantial degree of accuracy. This is a problem because it makes the feature matrix X singular and not invertible.

In [10]:
lr = LinearRegression()

data = np.array([x1, x2]).T

lr.fit(data, y)
lr.coef_, lr.intercept_

(array([0.4, 0.8]), 2.0)

We can see that the LinearRegression model from the scikit-learn library is able to handle this problem and still give us the optimal parameters. 

Sklearn's Linear Regression uses the non-negative least squares (nnls) function from scipy.optimize to solve the linear regression problem, the lsqr function from scipy.sparse.linalg which uses the Paige and Saunders algorithm which is an iterative algorithm to approximate the solution as well as the lstsq function from scipy.linalg for different scenarios. The LinearRegression function uses scipy.optimize.nnls in the case in which we want positive parameters, scipy.sparse.linalg.lsqr in case $X$(feature matrix) is sparse and in all other cases it uses scipy.linalg.lstsq function. When the feature matrix is singular, the optimal parameters cannot be calculated using the normal equation as shown above. For the cases of scipy.optimize.nnls and scipy.linalg.lstsq, the functions calculates the Moore-Penrose pseudo-inverse of the feature matrix and uses it to calculate the optimal parameters. The Moore-Penrose pseudo-inverse is calculated using the reduced singular value decomposition (SVD) of the feature matrix. The reduced SVD is a factorization of the feature matrix into three matrices $U$, $S$, and $V$ such that the feature matrix is equal to the product of $U$, $S$, and $V^T$. $U$ is of shape $(n,m)$, $V$ is of shape $(m,m)$ and $S$ is a diagonal matrix of shape $(m,m)$. The Moore-Penrose pseudo-inverse is calculated as the product of $V \cdot S^{-1} \cdot U^T$. The Moore-Penrose pseudo-inverse is then used to calculate the optimal parameters using the normal equation. The optimal parameters are calculated as the product of the Moore-Penrose pseudo-inverse and the target vector. The functions then uses the optimal parameters to calculate the predicted target vector. The predicted target vector is calculated as the product of the feature matrix and the optimal parameters. Hence, we do not get an error using the Linear Regression model when the feature matrix is singular. Following is how the Moore-Penrose pseudo-inverse is calculated using the reduced SVD of the feature matrix:

### Reduced SVD
$$
\begin{align*}
\text{X}&=U \cdot S \cdot V^T \\
U &\cdot U^T=I \\
V &\cdot V^T= V^T \cdot V = I \\
\theta&=(\text{X}^T \cdot \text{X})^{-1} \cdot \text{X}^T \cdot \text{y} \\
\theta&= ((V \cdot S^T \cdot U^T)\cdot (U \cdot S \cdot V^T))^{-1} \cdot V \cdot S^T \cdot U^T \cdot \text{y} \\
\theta&= (U \cdot S \cdot V^T)^{-1} \cdot (V \cdot S^T \cdot U^T)^{-1} \cdot V \cdot S^T \cdot U^T \cdot \text{y}  \\
\theta&= V \cdot S^{-1} \cdot U^{-1} \cdot (U^T)^{-1} \cdot (S^T)^{-1} \cdot V^{-1} \cdot V \cdot S^T \cdot U^T \cdot \text{y} \\
\theta&= V \cdot S^{-1} \cdot U^{T} \cdot \text{y} \\
\theta&= {\text{Moore-Penrose Pseudo-Inverse}} \cdot \text{y}
\end{align*}
$$